In [ ]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. Environment and Database Connection
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# 2. Fetch data from RAW table
df = pd.read_sql("SELECT * FROM raw_housing_data", engine)

def clean_data(df):
    """
    Integrates critical findings from EDA: Outlier removal and logical imputation.
    """
    df = df.copy()

    # A. OUTLIER REMOVAL
    # Çox böyük sahəsi olub qiyməti çox aşağı olan evlər modeli çaşdırır.
    if 'total_living_area' in df.columns and 'sale_price' in df.columns:
        outlier_indices = df[(df['total_living_area'] > 5000) & (df['sale_price'] < 400000)].index
        df = df.drop(outlier_indices, axis=0)

    # B. TECHNICAL DROPS
    df = df.drop(columns=['order_id', 'pid', 'garage_yr_blt'], errors='ignore')

    # C. LOGICAL IMPUTATION
    # Qarajı olmayan evlərə 'None' və 0 veririk
    garage_num_cols = [col for col in df.columns if 'garage' in col and df[col].dtype != 'object']
    garage_cat_cols = [col for col in df.columns if 'garage' in col and df[col].dtype == 'object']
    df[garage_num_cols] = df[garage_num_cols].fillna(0)
    df[garage_cat_cols] = df[garage_cat_cols].fillna('None')

    # Zirzəmisi olmayan evlərə 'None' və 0 veririk
    bsmt_num_cols = [col for col in df.columns if 'bsmt' in col and df[col].dtype != 'object']
    bsmt_cat_cols = [col for col in df.columns if 'bsmt' in col and df[col].dtype == 'object']
    df[bsmt_num_cols] = df[bsmt_num_cols].fillna(0)
    df[bsmt_cat_cols] = df[bsmt_cat_cols].fillna('None')

    # Digər kiçik boşluqlar (Electrical, Masonry)
    if 'electrical' in df.columns:
        df['electrical'] = df['electrical'].fillna(df['electrical'].mode()[0])
    if 'mas_vnr_area' in df.columns:
        df['mas_vnr_area'] = df['mas_vnr_area'].fillna(0)
    if 'mas_vnr_type' in df.columns:
        df['mas_vnr_type'] = df['mas_vnr_type'].fillna('None')

    # --- D. ADVANCED IMPUTATION (Lot Frontage) ---
    if 'neighborhood' in df.columns and 'lot_frontage' in df.columns:
        df['lot_frontage'] = df.groupby('neighborhood')['lot_frontage'].transform(
            lambda x: x.fillna(x.mean())
        )
    df['lot_frontage'] = df['lot_frontage'].fillna(df['lot_frontage'].mean())

    return df

# Prosesi icra edirik
df_cleaned = clean_data(df)

# 3. Save to SQL
df_cleaned.to_sql('cleaned_housing_data', engine, if_exists='replace', index=False)

print(f"Success: Processed {len(df_cleaned)} rows. Outliers removed and missing values handled.")